# 04 · Approach 3: Deep Learning with a Pre-Trained Transformer

Chapter 1 introduces deep learning for NLP: RNNs, LSTMs, CNNs and finally **transformers** with **transfer learning**:

1. **Pre-training:** a large neural network reads huge amounts of general text (Wikipedia, books) and learns *how language works*, without any sentiment labels.
2. **Fine-tuning:** we add a small classification layer on top and continue training on **our** labelled data for a few minutes.

The model starts with general knowledge of English (grammar, word meanings, typical contexts) and only has to learn the sentiment task itself.

We use **DistilBERT** (`distilbert-base-uncased`, Sanh et al. 2019): a smaller, 40% lighter version of BERT with about **66 million parameters**. It is small enough to fine-tune on a normal laptop CPU.

**Contents**
- **Part A:** fine-tune DistilBERT on our training data (the main experiment).
- **Part B (optional):** use a ready-made sentiment model that was trained on a *different* dataset, and see how well it transfers to IMDB (**domain shift**).

> ⏱️ **Runtime:** IMDB reviews are long, and transformers are expensive.
> - **CPU (most laptops):** by default the notebook fine-tunes on a random sample of **4,000** training reviews. This takes very roughly 30–60 minutes, depending on your machine.
> - **NVIDIA GPU:** it automatically uses all **22,500** training reviews (roughly 10–20 minutes).
> - For a quick test that everything works, set `TRAIN_LIMIT = 500` in the configuration cell.

In [1]:
# --- Make the project's `src` folder importable -----------------------------
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import random
import numpy as np
import torch                                         # PyTorch: the deep-learning framework
from torch.utils.data import DataLoader              # feeds the data in mini-batches
from tqdm.auto import tqdm                           # progress bars

from transformers import (
    AutoTokenizer,                       # loads the tokenizer that belongs to a model
    AutoModelForSequenceClassification,  # loads a model + a classification head
    get_linear_schedule_with_warmup,     # learning-rate schedule used for fine-tuning
)

from src.utils import (load_imdb, evaluate, print_report, save_results,
                       save_predictions, Timer, PROJECT_ROOT)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.15.0+cpu


## 0. Configuration
All settings in one place, so you can change them easily.

In [3]:
MODEL_NAME = "distilbert/distilbert-base-uncased"   # pre-trained model from the Hugging Face Hub
MAX_LENGTH = 256       # max tokens per review; longer reviews are cut (see section 1)
BATCH_SIZE = 16        # reviews processed together in one step (lower it to 8 if you run out of memory)
EPOCHS = 2             # full passes over the training data
LEARNING_RATE = 3e-5   # small steps: we only want to *adjust* the pre-trained weights
WARMUP_SHARE = 0.1     # first 10% of steps: learning rate grows slowly from 0
SEED = 42              # fixed random seed -> reproducible results
SAVE_MODEL = False     # True = save the fine-tuned model to models/ (about 250 MB, not pushed to GitHub)

# Choose the fastest available hardware:
#   "cuda" = NVIDIA GPU, "mps" = Apple Silicon GPU, otherwise the CPU.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Training on:", DEVICE)

# How many training reviews to use for fine-tuning.
# On a GPU we can afford all of them (None). On a CPU we use a random sample of
# 4,000, so that the notebook finishes in reasonable time. Notebook 03's learning
# curve also has a point at 4,000 reviews, so notebook 05 can still compare
# DistilBERT and Logistic Regression with the SAME amount of training data.
TRAIN_LIMIT = None if DEVICE.type == "cuda" else 4000
print("Training reviews used:", "all" if TRAIN_LIMIT is None else TRAIN_LIMIT)

Training on: cpu
Training reviews used: 4000


In [4]:
def set_seed(seed: int) -> None:
    """Fix all random number generators so that every run gives the same result."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

data = load_imdb()
train, val, test = data["train"], data["validation"], data["test"]
if TRAIN_LIMIT and TRAIN_LIMIT < len(train):
    # Same seed (42) as the learning curve in notebook 03, so for 4,000 reviews
    # both notebooks use exactly the same training sample.
    train = train.sample(n=TRAIN_LIMIT, random_state=SEED)
print(f"train {len(train):,} | validation {len(val):,} | test {len(test):,}")

train 4,000 | validation 2,500 | test 5,000


## 1. The tokenizer: how a transformer reads text

Transformers do not use whole words. They use **sub-word tokens** (DistilBERT uses *WordPiece*, with a vocabulary of about 30,000 pieces). A rare word is split into known pieces, so the model **never meets a completely unknown word**. That solves the long-tail problem from notebook 01.

The tokenizer also adds two special tokens:
- `[CLS]` at the start. Its final vector is used as a summary of the whole review for classification.
- `[SEP]` at the end, marking the end of the text.

In [5]:
# Downloads the tokenizer files the first time (a few hundred KB), then uses the local cache.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example = "An unforgettable, heartwarming masterpiece!"
encoded = tokenizer(example)

print("Text     :", example)
print("Tokens   :", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("Token IDs:", encoded["input_ids"])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

C:\Users\areft\Projects\practical-nlp-ch01-three-approaches\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\areft\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Text     : An unforgettable, heartwarming masterpiece!
Tokens   : ['[CLS]', 'an', 'un', '##for', '##get', '##table', ',', 'heart', '##war', '##ming', 'masterpiece', '!', '[SEP]']
Token IDs: [101, 2019, 4895, 29278, 18150, 10880, 1010, 2540, 9028, 6562, 17743, 999, 102]


Words the vocabulary does not contain are split into pieces that start with `##`, which means "continues the previous token". The model is *uncased*: it lowercases everything itself.

**How long are the reviews in tokens?** Let's count the tokens of every training review.

In [6]:
token_lengths = [len(ids) for ids in tokenizer(list(train["text"]))["input_ids"]]
print(f"Mean tokens per review: {np.mean(token_lengths):.0f}")
print(f"Median tokens per review: {np.median(token_lengths):.0f}")
print(f"Share of reviews longer than {MAX_LENGTH} tokens: {np.mean(np.array(token_lengths) > MAX_LENGTH):.1%}")
print(f"Share of reviews longer than 512 tokens (DistilBERT's hard limit): {np.mean(np.array(token_lengths) > 512):.1%}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (914 > 512). Running this sequence through the model will result in indexing errors


Mean tokens per review: 298
Median tokens per review: 219
Share of reviews longer than 256 tokens: 40.8%
Share of reviews longer than 512 tokens (DistilBERT's hard limit): 13.5%


A large share of the reviews is longer than `MAX_LENGTH`, and some are even longer than 512, the maximum DistilBERT can read at all. So we must **truncate**: we keep the first 256 tokens and drop the rest.

**Why 256 and not 512?** The cost of a transformer grows with the *square* of the input length, so 512 tokens is roughly 3–4 times slower than 256. Most reviews state their opinion clearly within the first 256 tokens, so we accept the small loss of information for a large gain in speed. This is a typical engineering trade-off.

> 💡 **Exercise:** research shows that keeping the **beginning and the end** of a long review (*head + tail truncation*) often works better than keeping only the beginning, because reviewers often state their verdict in the last sentences. Try it after finishing this notebook (Sun et al. 2019, *How to Fine-Tune BERT for Text Classification?*).

## 2. Feeding the data in mini-batches

A `DataLoader` splits the data into **mini-batches** of `BATCH_SIZE` reviews. The `collate` function turns a list of reviews into tensors:
- **truncation:** reviews longer than `MAX_LENGTH` tokens are cut;
- **padding:** shorter reviews are filled up with `[PAD]` tokens so that all reviews in a batch have the same length. We pad *per batch* ("dynamic padding"), which saves time when a batch contains only short reviews.
- **attention mask:** 1 for real tokens and 0 for padding, so the model ignores the padding.

In [7]:
def make_loader(df, shuffle: bool) -> DataLoader:
    """Create a DataLoader from a DataFrame with 'text' and 'label' columns."""
    # A list of (text, label) pairs is a valid PyTorch dataset: it has len() and indexing.
    pairs = list(zip(df["text"], df["label"]))

    def collate(batch):
        texts, labels = zip(*batch)                 # unzip into two tuples
        enc = tokenizer(
            list(texts),
            padding=True,              # pad to the longest review in this batch
            truncation=True,           # cut reviews longer than max_length
            max_length=MAX_LENGTH,
            return_tensors="pt",       # return PyTorch tensors
        )
        enc["labels"] = torch.tensor(labels)        # the model computes the loss itself if labels are given
        return enc

    # shuffle=True for training: a new random order every epoch helps learning
    return DataLoader(pairs, batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=collate)


train_loader = make_loader(train, shuffle=True)
val_loader = make_loader(val, shuffle=False)
test_loader = make_loader(test, shuffle=False)

# Look at one batch to understand the shapes
batch = next(iter(train_loader))
for key, tensor in batch.items():
    print(f"{key:<15} shape {tuple(tensor.shape)}")

input_ids       shape (16, 256)
token_type_ids  shape (16, 256)
attention_mask  shape (16, 256)
labels          shape (16,)


`input_ids` has shape *(batch size, longest review in the batch, at most MAX_LENGTH)*.

## 3. Load the pre-trained model

`AutoModelForSequenceClassification` loads the pre-trained DistilBERT body and adds a new, **randomly initialised** classification head (a small linear layer with 2 outputs). You will see a message that some weights are "newly initialized". This is expected: those are the weights of the head we are about to train.

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},   # stored with the model, makes outputs readable
    label2id={"negative": 0, "positive": 1},
)
model.to(DEVICE)   # move all weights to the GPU/CPU we chose

n_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {n_params / 1e6:.1f} million")

C:\Users\areft\Projects\practical-nlp-ch01-three-approaches\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\areft\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Number of parameters: 67.0 million


For comparison: our Logistic Regression in notebook 03 had one weight per TF-IDF feature, i.e. some tens of thousands. DistilBERT has about **1,000 times more** parameters.

## 4. Optimizer and learning-rate schedule

- **AdamW** is the standard optimizer for transformers: Adam with a correct implementation of *weight decay* (a mild regularisation).
- **Linear schedule with warm-up:** the learning rate rises from 0 during the first 10% of steps, then falls linearly back to 0. Starting gently avoids destroying the pre-trained knowledge with large random updates from the new head.

In [9]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_SHARE * total_steps),
    num_training_steps=total_steps,
)
print(f"{len(train_loader)} batches per epoch x {EPOCHS} epochs = {total_steps} training steps")

250 batches per epoch x 2 epochs = 500 training steps


## 5. Helper: predict a whole dataset

During evaluation we:
- switch the model to `eval()` mode, which turns off *dropout* (random switching-off of neurons that is only used during training);
- use `torch.no_grad()`, so PyTorch does not store the information needed for back-propagation. That is faster and uses less memory.

In [10]:
@torch.no_grad()   # decorator: no gradients are computed inside this function
def predict(loader: DataLoader) -> np.ndarray:
    """Return the predicted class (0/1) for every review in the loader."""
    model.eval()
    all_preds = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}   # move batch to the device
        logits = model(**batch).logits      # raw scores, shape (batch, 2)
        preds = logits.argmax(dim=-1)       # index of the larger score = predicted class
        all_preds.append(preds.cpu().numpy())
    return np.concatenate(all_preds)

## 6. The training loop

For every mini-batch:
1. **Forward pass:** the model predicts and computes the *cross-entropy loss* (how wrong it is).
2. **Backward pass:** `loss.backward()` computes the gradient of the loss with respect to each of the 66 million weights.
3. **Update:** `optimizer.step()` moves every weight a small step in the direction that reduces the loss.
4. `optimizer.zero_grad()` resets the gradients, because PyTorch adds them up by default.

After each epoch we check the accuracy on the **validation** set, to see whether the model is still improving.

In [11]:
history = []
with Timer() as t_train:
    for epoch in range(1, EPOCHS + 1):
        model.train()                      # turn dropout on (training mode)
        running_loss = 0.0
        progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

        for batch in progress:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            outputs = model(**batch)       # 1. forward pass (loss included, since we passed labels)
            loss = outputs.loss

            loss.backward()                # 2. backward pass: compute gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # avoid exploding gradients
            optimizer.step()               # 3. update the weights
            scheduler.step()               #    update the learning rate
            optimizer.zero_grad()          # 4. reset gradients for the next batch

            running_loss += loss.item()
            progress.set_postfix(loss=f"{loss.item():.3f}")   # show current loss in the bar

        avg_loss = running_loss / len(train_loader)
        val_acc = (predict(val_loader) == val["label"].values).mean()
        history.append({"epoch": epoch, "train_loss": avg_loss, "val_accuracy": val_acc})
        print(f"Epoch {epoch}: average training loss {avg_loss:.4f} | validation accuracy {val_acc:.4f}")

print(f"\nTotal training time: {t_train.seconds / 60:.1f} minutes")

Epoch 1/2:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 1: average training loss 0.3975 | validation accuracy 0.8796


Epoch 2/2:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 2: average training loss 0.2013 | validation accuracy 0.8824

Total training time: 44.5 minutes


The training loss should go down from epoch to epoch. If the training loss keeps falling but the validation accuracy starts to drop, the model is **overfitting** (memorising the training data). For DistilBERT on a dataset of this size, 2–3 epochs is usually the sweet spot.

## 7. Final evaluation on the test set

In [ ]:
with Timer() as t_pred:
    test_pred_dl = predict(test_loader)

metrics_dl = evaluate(test["label"], test_pred_dl)
print_report("Fine-tuned DistilBERT (test set)", metrics_dl)

save_results(
    "dl_distilbert_finetuned", metrics_dl,
    family="deep_learning",
    n_train_examples=len(train),
    train_seconds=t_train.seconds,
    predict_seconds=t_pred.seconds,
    device=str(DEVICE),
    model=MODEL_NAME, epochs=EPOCHS, learning_rate=LEARNING_RATE,
)
save_predictions("dl_distilbert_finetuned", test["text"], test["label"], test_pred_dl);

if SAVE_MODEL:
    out_dir = PROJECT_ROOT / "models" / "distilbert-imdb"
    model.save_pretrained(out_dir)       # weights + config
    tokenizer.save_pretrained(out_dir)   # tokenizer files
    print("Model saved to", out_dir)

## 8. Try it yourself
Write your own mini-reviews, including tricky ones with negation, contrast or irony.

In [ ]:
@torch.no_grad()
def classify(text: str) -> None:
    """Print the predicted class and the model's confidence for one text."""
    model.eval()
    enc = tokenizer(text, return_tensors="pt", truncation=True,
                    max_length=MAX_LENGTH).to(DEVICE)
    probs = torch.softmax(model(**enc).logits, dim=-1)[0]   # logits -> probabilities
    label = int(probs.argmax())
    print(f"{model.config.id2label[label]:>8} ({probs[label]:.0%})  {text}")

for s in [
    "The film is not good.",
    "It isn't boring at all, I enjoyed every minute.",
    "The actors are good, but the script is a boring mess.",
    "I wanted to love this movie. I really did.",
    "A two-hour nap would have been more exciting.",
]:
    classify(s)

## Part B (optional): a ready-made model and domain shift

Instead of fine-tuning ourselves, we can download a model that someone else has **already fine-tuned for sentiment**. A very popular one is `distilbert-base-uncased-finetuned-sst-2-english`. It was fine-tuned on **SST-2** (Stanford Sentiment Treebank), which consists of **single sentences from professional critics' reviews on Rotten Tomatoes**. It was not trained on IMDB data.

So this is a test of **domain shift**, one of the reasons the book gives in Chapter 1 for why *"deep learning is not yet the silver bullet"*: a model trained on one kind of text can lose accuracy on another. Here the task is the same (movie sentiment), but:
- **source:** professional critics vs. ordinary IMDB users;
- **length:** one sentence vs. a review of several paragraphs.

We give it the same 256-token input as our own model and compare. No IMDB labels are used at all, so, like the heuristics, it needs **0** labelled examples from our dataset.

In [ ]:
RUN_PART_B = True   # set to False to skip (downloads another ~250 MB model)

if RUN_PART_B:
    from transformers import pipeline

    sst2_classifier = pipeline(
        "text-classification",
        model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
        device=DEVICE,
    )
    with Timer() as t_pred_b:
        # truncation + max_length: cut long reviews exactly like for our own model
        outputs = sst2_classifier(list(test["text"]), batch_size=BATCH_SIZE,
                                  truncation=True, max_length=MAX_LENGTH)
    # The pipeline returns e.g. {'label': 'POSITIVE', 'score': 0.99} for every review.
    test_pred_sst2 = np.array([1 if o["label"].upper().startswith("POS") else 0 for o in outputs])

    metrics_sst2 = evaluate(test["label"], test_pred_sst2)
    print_report("Ready-made SST-2 model, no IMDB training (test set)", metrics_sst2)

    save_results(
        "dl_sst2_pretrained", metrics_sst2,
        family="deep_learning",
        n_train_examples=0,              # no IMDB labels used
        train_seconds=0.0,
        predict_seconds=t_pred_b.seconds,
        model="distilbert-base-uncased-finetuned-sst-2-english",
    )
    save_predictions("dl_sst2_pretrained", test["text"], test["label"], test_pred_sst2)

Compare this score with your fine-tuned model from Part A and with the heuristics from notebook 02:
- If the ready-made model is clearly **below** your fine-tuned model, that difference is the price of domain shift. A few thousand in-domain examples were worth more than a model trained on a lot of *other* data.
- If it is **close**, the two domains are similar enough that the knowledge transfers well. That is useful in practice: when you have no labels yet, a related ready-made model can be a much stronger starting point than hand-written rules.

## Summary: deep learning

| | |
|---|---|
| ✅ **Strengths** | Usually the highest accuracy · understands word order, negation and context much better · sub-word tokens mean no unknown words · transfer learning needs fewer labels than training from scratch |
| ❌ **Weaknesses** | Much slower to train and to run · long texts must be truncated · needs more memory (a GPU in production) · hard to interpret (a black box) · can lose accuracy when the text domain changes |
| 💡 **When to use** | When accuracy really matters and you have the compute budget, or when context and word order are essential to the task |

The book's section *"Why Deep Learning Is Not Yet the Silver Bullet for NLP"* lists more caveats: overfitting on small datasets, domain adaptation, interpretability, cost, and on-device deployment.

**Next:** `05_comparison.ipynb`, all approaches side by side.